## 0. 제출 정보
- 이름: 우다현
- GitHub ID: woodahyun
- 작성일: 2026-09-28
- 최종 제출 URL: https://github.com/woodahyun/llm-data-analysis-study/blob/main/chapter04/chapter04.ipynb

## 1. 질문과 필요한 데이터 선택
### 내가 확인하려는 질문
completed 된 주문을 기준으로 어떤 상품 카테고리와 상품이 매출에 많이 기여하는지, 월별 매출과 고객별 구매금액은 어떻게 분포하는지 확인한다.

### 사용한 파일/컬럼
- orders.csv: order_id, customer_id, order_date, order_status
- order_items.csv: order_id, product_id, quantity, unit_price
- products.csv: product_id, product_name, category
- customers.csv: customer_id, city

### 결과 관찰
주문상세 데이터를 기준으로 수량과 실제 판매단가를 곱해 주문상세 금액을 계산하고, 주문 상태가 completed인 주문만 분석에 포함했다. 완료 주문은 184건, 완료 주문상세 행은 474건이었으며 완료 주문 기준 계산 매출은 148,990,000원이었다.

### 나의 해석과 판단
매출 분석에는 단순 주문 수가 아니라 실제 판매된 상품의 수량과 판매단가가 필요하므로 order_items의 quantity, unit_price를 선택했다. 또한 취소·환불 주문이 포함되면 실제 소비를 과대평가할 수 있으므로 order_status == 'completed'를 분석 범위로 정했다. 카테고리와 상품별 결과를 보기 위해 상품 정보를 결합했고, 월별·고객별 분석을 위해 주문일과 고객 ID를 사용했다.

### 업무·분석적 의미
이 질문을 통해 어떤 카테고리·상품이 매출에 크게 기여하는지, 매출이 어느 시기에 집중되는지, 어떤 고객이 높은 구매금액을 보이는지 확인할 수 있다. 이는 상품 재고·프로모션 우선순위, 월별 판매계획, 고객 세분화의 기초 자료로 활용할 수 있다.

### 한계와 추가 확인 사항
현재 데이터에는 할인, 쿠폰, 배송비, 세금, 반품 완료 시점, 결제수수료 정보가 없다. 따라서 계산한 매출은 주문상세의 판매금액 합계이며 회계상 순매출과 같다고 단정할 수 없다. 또한 고객별 구매금액 차이의 원인이나 특정 상품이 많이 팔린 이유는 현재 데이터만으로 알 수 없다.

## 2. 필터링·정렬·파생 컬럼
- 적용한 필터 조건:
고객 예시 필터는 age >= 30, city == '서울', city.isin(['서울', '부산'])를 사용했다. 핵심 매출 분석에서는 order_status == 'completed'를 적용했다.

- 정렬 기준:
상품 가격은 price 오름차순·내림차순으로 확인했고, 카테고리별·상품별·고객별 결과는 total_sales 내림차순으로 정렬했다. 월별 결과는 order_month 오름차순으로 정렬했다.

- 만든 파생 컬럼:
line_total, order_month, customer_label

- `line_total` 계산식:
line_total = quantity * unit_price

![필터와 파생 컬럼](images/step02_transform.png)

### 결과 관찰
30세 이상 고객은 111명, 서울 고객은 15명, 서울 또는 부산 고객은 31명이었다. 주문상세 전체 764행의 line_total 합계는 255,610,000원이었으나, 완료 주문만 필터링한 뒤의 합계는 148,990,000원이었다. 즉 취소·환불 주문을 제외하면 전체 주문상세 금액보다 106,620,000원이 감소했다.

### 나의 해석과 판단
필터 기준이 달라지면 결과도 달라진다. 예를 들어 전체 주문상세 금액을 사용하면 취소·환불 주문까지 포함되어 매출이 실제보다 크게 계산된다. 반대로 완료 주문만 사용하면 실제 구매가 완료된 주문 중심의 매출을 볼 수 있다. 연령이나 도시 필터를 적용하면 해당 조건의 고객 집단만 남으므로, 전체 고객의 특성으로 일반화하면 안 된다.

### 업무·분석적 의미
필터링과 파생 컬럼은 분석 기준을 명확하게 만드는 과정이다. 특히 line_total을 주문상세 수준에서 먼저 계산하면 이후 카테고리·상품·월·고객 기준으로 같은 매출 정의를 일관되게 집계할 수 있다.

### 한계와 추가 확인 사항
unit_price가 할인·쿠폰 적용 후 실제 결제단가인지 확인이 필요하다. 또한 주문 상태가 completed라도 이후 반품이나 부분환불이 발생했는지 알 수 없으며, 주문일이 결제일·배송일·주문생성일 중 무엇인지도 확인해야 한다.

## 3. merge 검증
- 병합한 데이터:
1. order_items와 orders 병합
2. 완료 주문상세와 products 병합
3. 고객별 매출과 customers 병합

- 사용한 key:
1. order_id
2. product_id
3. customer_id

- `validate` 결과:
order_items-orders 병합과 완료 주문상세-products 병합은 many_to_one 검증을 통과했다. 고객별 매출-고객 정보 병합은 one_to_one 검증을 통과했다.

- `indicator` 결과:
주문 병합은 764행 모두 both, 상품 병합은 완료 주문상세 474행 모두 both였다. left_only와 right_only는 0건이었다.

- 병합 전/후 행 수:
order_items-orders 병합은 764행 → 764행, 완료 주문상세-products 병합은 474행 → 474행이었다.

![merge 검증](images/step03_merge.png)

### 결과 관찰
주문 ID와 상품 ID가 모두 기준 테이블에 존재했고, 병합 전후 행 수가 변하지 않았다. 따라서 주문상세 행에 주문 상태·주문일·고객 ID와 상품명·카테고리를 안전하게 추가할 수 있었다.

### 나의 해석과 판단
이번 병합은 주문상세 여러 행이 하나의 주문 또는 하나의 상품에 연결되는 구조이므로 many_to_one이 적절하다. validate 검증을 통과했고 indicator 결과가 모두 both이며 행 수도 유지되었으므로, 중복 키로 인해 매출이 불필요하게 늘어나거나 연결되지 않은 주문상세가 누락될 위험이 낮다고 판단했다.

### 업무·분석적 의미
잘못된 merge는 동일 주문상세가 여러 번 복제되어 매출이 과대계산되거나, 연결되지 않은 행이 빠져 매출이 과소계산되는 문제를 만들 수 있다. 따라서 병합 전에 키의 중복 여부를 확인하고, 병합 후에는 행 수와 _merge 결과를 확인해야 한다.

### 한계와 추가 확인 사항
현재 검증은 ID가 존재하고 관계의 형태가 맞는지를 확인한 것이다. ID가 업무적으로 올바른 고객·상품과 연결되었는지, 상품명이나 카테고리 값 자체가 정확한지는 별도 확인이 필요하다. 또한 이후 새로운 데이터가 추가될 때도 같은 검증을 반복해야 한다.

## 4. completed 주문 범위와 집계
- 분석 범위 정의:
order_status == 'completed'인 주문만 포함했다. 매출은 주문상세별 quantity × unit_price의 합계로 정의했다.

- 카테고리별 결과:
스포츠가 31,743,000원(21.31%)으로 가장 높았고, 패션이 10,587,000원(7.11%)으로 가장 낮았다.

- 상품별 결과:
스포츠 상품 041이 5,705,000원으로 가장 높은 매출을 기록했다.

- 월별 결과:
2025년 10월 매출이 25,766,000원으로 가장 높았고, 완료 주문 수는 26건이었다. 2026년 7월은 2,188,000원으로 가장 낮았으나 월 전체가 아닌 일부 기간 데이터이다.

- 고객별 결과:
익명화한 Customer 117이 4,100,000원으로 가장 높은 구매금액을 기록했고, 완료 주문 수는 5건이었다.

![핵심 집계 결과](images/step04_groupby.png)

### 결과 관찰
카테고리별 매출 상위 3개는 스포츠 31,743,000원, 전자기기 26,400,000원, 생활용품 23,915,000원이었다. 상위 상품은 스포츠 상품 041로 35개가 판매되어 5,705,000원의 매출을 기록했다. 월별로는 2025년 10월이 가장 높은 매출을 보였으며, 고객별로는 상위 고객의 구매금액과 주문 수를 함께 확인할 수 있었다.

### 나의 해석과 판단
스포츠 카테고리가 전체 완료 주문 매출의 21.31%를 차지한 점이 가장 중요하다고 판단했다. 단순히 판매 수량뿐 아니라 매출 기여도가 가장 높으므로, 재고 관리나 관련 프로모션 우선순위를 정할 때 먼저 검토할 수 있는 카테고리이기 때문이다.

### 업무·분석적 의미
카테고리·상품별 결과는 재고 및 프로모션 우선순위를 정하는 데 활용할 수 있다. 월별 결과는 판매계획과 인력·재고 배치의 기초 자료가 될 수 있으며, 고객별 결과는 재구매 고객이나 고액 구매 고객을 추가 분석하는 출발점이 된다.

### 한계와 추가 확인 사항
total_sales는 quantity × unit_price의 합계일 뿐이므로 회계상 순매출과 같다고 단정할 수 없다. 배송비, 세금, 할인·쿠폰, 결제수수료, 부분취소, 반품, 이후 환불 정보가 반영되지 않았을 수 있기 때문이다. 또한 2025년 7월과 2026년 7월은 부분 월 데이터이므로 다른 완전한 월과 단순 비교하면 안 된다.

## 5. 총합 일치 검증
- 원본 completed `line_total` 합계: 148,990,000원
- 카테고리 합계: 148,990,000원
- 월별 합계: 148,990,000원
- 고객별 합계: 148,990,000원
- 차이 여부: 차이 없음

![총합 검증](images/step05_total_check.png)

### 나의 해석과 판단
- 총합 검증은 같은 완료 주문 데이터를 서로 다른 기준으로 집계했을 때 매출이 빠지거나 중복되지 않았는지 확인하는 과정이다. 이번에는 네 합계가 모두 일치하므로 카테고리·월·고객별 집계가 동일한 원본 완료 주문상세 금액을 보존했다고 판단했다.
- 불일치가 발생하면 먼저 완료 주문 필터가 모든 집계에 동일하게 적용되었는지 확인하고, 다음으로 병합 뒤 행 수가 증가했는지, 결측 카테고리·고객 ID가 있는지, 그룹화 과정에서 제외된 행이 있는지 순서대로 확인해야 한다.

## 6. LLM pandas 코드 검증
- LLM Prompt 요약:
order_items.groupby('category')['line_total'].sum() 코드가 현재 데이터에서 바로 실행 가능한지 검토하고, 카테고리 컬럼의 위치, 주문 상태 필터링, merge의 validate와 indicator를 포함한 안전한 수정 코드와 검증 순서를 요청했다.

- 제안 코드 요약:
order_items를 카테고리 기준으로 그룹화한 뒤 line_total을 합산하는 코드였다.

- 실제 컬럼/범위와 맞지 않은 부분:
category 컬럼은 order_items가 아니라 products에 있다. line_total도 원래 데이터에 있는 컬럼이 아니라 quantity × unit_price로 먼저 만들어야 한다. 또한 주문 상태는 orders에 있으므로, 완료 주문 기준 매출을 보려면 주문 데이터를 병합하고 order_status == 'completed' 필터를 적용해야 한다. 원래 코드는 병합 검증도 포함하지 않았다.

- 수정한 내용:
주문상세에 line_total을 생성한 뒤, orders를 order_id 기준으로 병합했다. 이때 validate='many_to_one'과 indicator=True로 관계와 연결 결과를 검증했다. 완료 주문만 필터링한 뒤, products를 product_id 기준으로 다시 병합하여 category를 추가하고 카테고리별 매출을 집계했다.

- 최종 판단: 수정 후 사용

![LLM 코드 검증](images/step06_llm_validation.png)

### 나의 해석과 판단
- LLM 코드가 오류 없이 실행되었다고 해서 분석이 맞는 것은 아니다. 코드가 실행되더라도 분석에 필요한 완료 주문 필터가 빠져 있거나, 잘못된 파일의 컬럼을 사용하거나, 병합 과정에서 행이 중복될 수 있다. 이번 사례에서도 원래 코드는 category의 실제 위치와 주문 상태 필터를 반영하지 않았으므로, 그대로 사용했다면 오류가 발생하거나 전체 주문 기준의 다른 결과가 나왔을 것이다.
- 따라서 LLM이 제안한 코드는 실제 컬럼명, 데이터 범위, 병합 관계, 집계 후 총합을 사람이 직접 확인한 뒤 사용해야 한다.

## 7. Chapter 04 최종 인사이트
### 가장 의미 있다고 생각한 결과 2가지
1. 완료 주문 매출 기준 스포츠 카테고리가 31,743,000원으로 가장 높았고, 전체 매출의 21.31%를 차지했다.
2. 월별 매출은 2025년 10월에 25,766,000원으로 가장 높았으며, 같은 달 완료 주문 수도 26건으로 가장 많았다.

### 그 결과를 뒷받침하는 수치/표
구분 | 결과 ||
완료 주문 매출 합계 | 148,990,000원 ||
스포츠 카테고리 매출 | 31,743,000원 ||
스포츠 매출 비중 | 21.31% ||
2025년 10월 매출 | 25,766,000원 ||
2025년 10월 완료 주문 수 | 26건 ||
최고 매출 상품 | 스포츠 상품 041, 5,705,000원 ||
최고 구매금액 고객 | Customer 117, 4,100,000원

### 추가로 확인하고 싶은 질문
스포츠 카테고리의 높은 매출이 특정 고가 상품, 높은 판매 수량, 특정 고객 집단, 또는 특정 시기의 프로모션 때문인지 추가로 확인하고 싶다. 또한 완료 주문과 취소·환불 주문의 상품·결제수단별 차이도 분석하고 싶다.

### 현재 결과의 한계
이번 분석은 샘플 데이터의 주문상세 금액을 기준으로 한 집계 결과이다. 할인·쿠폰·반품·배송비·세금·결제수수료 정보가 없으므로 실제 회계상 순매출이나 수익성을 판단할 수 없다. 또한 월별 데이터의 시작과 종료가 부분 월이므로 계절성이나 월별 추세를 확정적으로 해석하기 어렵다.

## 최종 제출 체크
- [o] 핵심 셀 Output이 남아 있습니다.
- [o] merge와 총합 검증 Evidence가 있습니다.
- [o] 결과 관찰과 해석이 구분되어 있습니다.
- [o] LLM 코드를 검증했습니다.
- [o] 개인정보/Secret이 없습니다.
- [o] `chapter04/chapter04.ipynb`가 GitHub에서 정상 표시됩니다.
- [o] 최종 Notebook 파일 URL을 제출합니다.